# Stage 3 · Tissue-cluster-level refinement → refined anchors

Using the preliminary clusters, re-cluster at the tissue-cluster level with geosketch downsampling and leiden, re-scoring anchors (`dist_pc`) to produce **refined** anchors. Example tissue: **TrCo**.

These are the original pipeline notebooks, concatenated in execution order with paths normalized to `ENTEX_ROOT`. They document the method and run per tissue / major type across the full raw dataset (heavy compute); they are shown for reference and are **not re-executed in the book**. Example group shown where templated.

In [1]:
# === Reproduction setup ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT); import repro_guard


## 3a · geosketch downsampling (3C)

_Source: `clustering/tissue/L1/TrCo/02b.100k3C_geosketch.ipynb`_

In [2]:
# Parameters
cpu = 5
group_name = "TrCo"
mem_gb = 20


In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [4]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


## Downsample with geosketch

In [5]:
mcad = anndata.read_h5ad('HiC_100kb_pca.h5ad')

In [6]:
npc = significant_pc_test(mcad, p_cutoff=0.05, obsm='100k3C_pca', update=False)
mcad.obsm['X_pca'] = normalize(mcad.obsm['100k3C_pca'][:, :npc], axis=1)


In [7]:
from geosketch import gs

sketch_index = gs(mcad.obsm['X_pca'], mcad.shape[0]//3, replace=False)
# sketch_index = gs(mcad.X, mcad.shape[0]//3, replace=False)


In [8]:
selc = np.zeros(mcad.shape[0])
selc[sketch_index] = 1
mcad.obs['geosketch'] = selc.astype(int)


In [9]:
mcad.obs.iloc[sketch_index]['celltype'].value_counts()

In [10]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300, constrained_layout=True)

tmp = mcad.obs.iloc[sketch_index].copy()
ax = axes[0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds*4,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

tmp = mcad.obs.copy()
ax = axes[1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)



In [11]:
model = TruncatedSVD(n_components=50, algorithm='arpack')


In [12]:
model.fit(mcad.X[sketch_index])
mcad.obsm['100k3C_geosk_all'] = model.transform(mcad.X)
mcad.uns['100k3C_geosk_eigen'] = model.singular_values_


In [13]:
npc = significant_pc_test(mcad, p_cutoff=0.05, obsm='100k3C_geosk_all', update=False)


In [14]:
mcad.obsm['X_geosk_pca'] = normalize(mcad.obsm['100k3C_geosk_all'][:, :npc], axis=1)
tsne(mcad, obsm='X_geosk_pca', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_geosk_pc{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [15]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_geosk_pc{npc}_{coord_base}'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)


ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


## Compute anchor distance in PC space

In [16]:
sample_list = np.sort(mcad.obs['Donor'].unique())

In [17]:
anchor_file = f'HiC_100kb_seurat_anchor_{"_".join(sample_list)}.hdf'
anchor = pd.read_hdf(anchor_file)


In [18]:
coord1 = mcad.obsm['X_geosk_pca'][mcad.obs['Donor']==sample_list[0]][anchor['x1']]
coord2 = mcad.obsm['X_geosk_pca'][mcad.obs['Donor']==sample_list[1]][anchor['x2']]
anchor['dist_geosk_pc'] = np.linalg.norm((coord1 - coord2), ord=2, axis=1)


## Integrate with geosketch distance on PC space

In [19]:
thres1 = np.percentile(anchor['dist_geosk_pc'], 80)
thres2 = np.percentile(anchor['dist_geosk_pc'], 60)


In [20]:
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]

integrator = SeuratIntegration()
integrator.adata_dict = {k: v for k, v in zip(list(range(len(adata_list))), adata_list)}
integrator.n_dataset = len(adata_list)
integrator.n_cells = [adata.shape[0] for adata in adata_list]

# intra-dataset KNN for scoring the anchors
integrator.k_local = None
integrator.key_local = 'X_geosk_pca'
integrator._calculate_local_knn()

# alignments and all_pairs
integrator.alignments = None
integrator._get_all_pairs()
# integrator.anchor = anchor_df


In [21]:
integrator.anchor[(0,1)] = anchor.loc[anchor['dist_geosk_pc']<thres1, ['x1','x2','score']].copy()
corrected = integrator.integrate(key_correct='X_geosk_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [22]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}geofilter80'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}geofilter80'] = normalize(mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}geofilter80'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_pc{npc}_seuratcc{npc}geofilter80', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}geofilter80_tsne'] = mcad.obsm['X_tsne'].copy()


In [23]:
integrator.anchor[(0,1)] = anchor.loc[anchor['dist_geosk_pc']<thres2, ['x1','x2','score']].copy()
corrected = integrator.integrate(key_correct='X_geosk_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [24]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}geofilter60'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}geofilter60'] = normalize(mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}geofilter60'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_pc{npc}_seuratcc{npc}geofilter60', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}geofilter60_tsne'] = mcad.obsm['X_tsne'].copy()


In [25]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}geofilter60_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['CisLongContact']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [26]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}geofilter80_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['CisLongContact']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


## Compute anchor distance in U space

In [27]:
mcad.obsm['X_geosk_pca'] = normalize(mcad.obsm['100k3C_geosk_all'][:, :npc] / mcad.uns['100k3C_geosk_eigen'][:npc], axis=1)


In [28]:
coord1 = mcad.obsm['X_geosk_pca'][mcad.obs['Donor']==sample_list[0]][anchor['x1']]
coord2 = mcad.obsm['X_geosk_pca'][mcad.obs['Donor']==sample_list[1]][anchor['x2']]
anchor['dist_geosk_u'] = np.linalg.norm((coord1 - coord2), ord=2, axis=1)


## Integrate with geosketch distance on U space

In [29]:
thres1 = np.percentile(anchor['dist_geosk_u'], 80)
thres2 = np.percentile(anchor['dist_geosk_u'], 60)


In [30]:
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]

integrator = SeuratIntegration()
integrator.adata_dict = {k: v for k, v in zip(list(range(len(adata_list))), adata_list)}
integrator.n_dataset = len(adata_list)
integrator.n_cells = [adata.shape[0] for adata in adata_list]

# intra-dataset KNN for scoring the anchors
integrator.k_local = None
integrator.key_local = 'X_pca'
integrator._calculate_local_knn()

# alignments and all_pairs
integrator.alignments = None
integrator._get_all_pairs()
# integrator.anchor = anchor_df


In [31]:
integrator.anchor[(0,1)] = anchor.loc[anchor['dist_geosk_u']<thres1, ['x1','x2','score']].copy()
corrected = integrator.integrate(key_correct='X_geosk_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [32]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}geofilter80'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}geofilter80'] = normalize(mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}geofilter80'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_u{npc}_seuratcc{npc}geofilter80', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}geofilter80_tsne'] = mcad.obsm['X_tsne'].copy()


In [33]:
integrator.anchor[(0,1)] = anchor.loc[anchor['dist_geosk_u']<thres2, ['x1','x2','score']].copy()
corrected = integrator.integrate(key_correct='X_geosk_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [34]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}geofilter60'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}geofilter60'] = normalize(mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}geofilter60'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_u{npc}_seuratcc{npc}geofilter60', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}geofilter60_tsne'] = mcad.obsm['X_tsne'].copy()


In [35]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}geofilter60_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['CisLongContact']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [36]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}geofilter80_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['CisLongContact']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [37]:
# anchor.to_hdf(anchor_file, key='data')


In [38]:
anchor

In [39]:
fig, axes = plt.subplots(1, 2, figsize=(5,2), dpi=300)
ax = axes[0]
xx, yy = anchor['dist_geosk_u'], anchor['dist_geosk_pc']
ax.scatter(xx, yy, c='k', edgecolor='none', s=0.5, alpha=1, rasterized=True)
ax.plot([np.min(xx), np.max(xx)], [np.min(xx), np.max(xx)])
ax.set_ylabel('Dist Geosk PC')
ax.set_xlabel('Dist Geosk U')

ax = axes[1]
xx, yy = anchor['dist_pc'], anchor['dist_geosk_pc']
ax.scatter(xx, yy, c='k', edgecolor='none', s=0.5, alpha=1, rasterized=True)
ax.plot([np.min(xx), np.max(xx)], [np.min(xx), np.max(xx)])
ax.set_xlabel('Dist PC')
# ax.set_ylabel('Dist Geosk PC')


In [40]:
# mcad = anndata.AnnData(obs=mcad.obs, obsm=mcad.obsm)
mcad.write_h5ad('HiC_100kb_pca.h5ad')


In [41]:
mcad

## 3b · leiden refinement + anchor re-scoring

_Source: `clustering/tissue/L1/TrCo/03b.100k3C_leiden.ipynb`_

In [42]:
# Parameters
cpu = 5
group_name = "TrCo"
mem_gb = 20


In [43]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [44]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


## Downsample with Leiden

In [45]:
mcad = anndata.read_h5ad('HiC_100kb_pca.h5ad')

In [46]:
npc = significant_pc_test(mcad, p_cutoff=0.05, obsm='100k3C_pca', update=False)
mcad.obsm['X_pca'] = normalize(mcad.obsm['100k3C_pca'][:, :npc], axis=1)


In [47]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_tsne'].copy()
dump_embedding(mcad, coord_base)


In [48]:
sample_list = np.sort(mcad.obs['Donor'].unique())
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]


In [49]:
for adata in adata_list:
    sc.pp.neighbors(adata, n_neighbors=25, use_rep='X_pca')
    sc.tl.leiden(adata, resolution=1.0, random_state=0)


In [50]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300, constrained_layout=True)

for k,adata in enumerate(adata_list):
    ax = axes[k]
    ax.scatter(mcad.obs['tsne_0'], mcad.obs['tsne_1'], s=1, edgecolor='none', rasterized=True, c=(0.8, 0.8, 0.8), alpha=0.5)
    _ = categorical_scatter(data=adata.obs,
                            ax=ax,
                            coord_base=coord_base,
                            hue='leiden',
                            text_anno='leiden', 
                            s=ds*4,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            legend_kws={'ncol':1}, 
                            show_legend=True)


In [51]:
ncell = 100
selc = []
for adata in adata_list:
    for _,df in adata.obs.groupby('leiden'):
        if df.shape[0]>ncell:
            np.random.seed(0)
            selc.append(np.random.choice(df.index, ncell, False))
        else:
            selc.append(df.index)

selc = np.concatenate(selc)
print(selc.shape[0])


In [52]:
mcad.obs.loc[selc, 'celltype'].value_counts()

In [53]:
mcad.obs['ds_leiden'] = 0
mcad.obs.loc[selc, 'ds_leiden'] = 1


In [54]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300, constrained_layout=True)

tmp = mcad.obs.loc[selc].copy()
ax = axes[0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds*4,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

tmp = mcad.obs.copy()
ax = axes[1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)



In [55]:
model = TruncatedSVD(n_components=50, algorithm='arpack')


In [56]:
model.fit(mcad[selc].X)
mcad.obsm['100k3C_leiden_all'] = model.transform(mcad.X)
mcad.uns['100k3C_leiden_eigen'] = model.singular_values_


In [57]:
npc = significant_pc_test(mcad, p_cutoff=0.05, obsm='100k3C_leiden_all', update=False)


In [58]:
mcad.obsm['X_leiden_pca'] = normalize(mcad.obsm['100k3C_leiden_all'][:, :npc], axis=1)
tsne(mcad, obsm='X_leiden_pca', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_leiden_pc{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [59]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_leiden_pc{npc}_{coord_base}'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


## Compute anchor distance in PC space

In [60]:
anchor_file = f'HiC_100kb_seurat_anchor_{"_".join(sample_list)}.hdf'
anchor = pd.read_hdf(anchor_file)


In [61]:
coord1 = mcad.obsm['X_leiden_pca'][mcad.obs['Donor']==sample_list[0]][anchor['x1']]
coord2 = mcad.obsm['X_leiden_pca'][mcad.obs['Donor']==sample_list[1]][anchor['x2']]
anchor['dist_leiden_pc'] = np.linalg.norm((coord1 - coord2), ord=2, axis=1)


## Integrate with leiden distance in PC space

In [62]:
thres1 = np.percentile(anchor['dist_leiden_pc'], 80)
thres2 = np.percentile(anchor['dist_leiden_pc'], 60)


In [63]:
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]

integrator = SeuratIntegration()
integrator.adata_dict = {k: v for k, v in zip(list(range(len(adata_list))), adata_list)}
integrator.n_dataset = len(adata_list)
integrator.n_cells = [adata.shape[0] for adata in adata_list]

# intra-dataset KNN for scoring the anchors
integrator.k_local = None
integrator.key_local = 'X_leiden_pca'
integrator._calculate_local_knn()

# alignments and all_pairs
integrator.alignments = None
integrator._get_all_pairs()
# integrator.anchor = anchor_df


In [64]:
integrator.anchor[(0,1)] = anchor.loc[anchor['dist_leiden_pc']<thres1, ['x1','x2','score']].copy()
corrected = integrator.integrate(key_correct='X_leiden_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [65]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}ldnfilter80'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}ldnfilter80'] = normalize(mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}ldnfilter80'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_pc{npc}_seuratcc{npc}ldnfilter80', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}ldnfilter80_tsne'] = mcad.obsm['X_tsne'].copy()


In [66]:
integrator.anchor[(0,1)] = anchor.loc[anchor['dist_leiden_pc']<thres2, ['x1','x2','score']].copy()
corrected = integrator.integrate(key_correct='X_leiden_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [67]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}ldnfilter60'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}ldnfilter60'] = normalize(mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}ldnfilter60'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_pc{npc}_seuratcc{npc}ldnfilter60', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}ldnfilter60_tsne'] = mcad.obsm['X_tsne'].copy()


In [68]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}ldnfilter60_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['CisLongContact']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [69]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}ldnfilter80_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['CisLongContact']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


## Compute anchor distance in U space

In [70]:
mcad.obsm['X_leiden_pca'] = normalize(mcad.obsm['100k3C_leiden_all'][:, :npc] / mcad.uns['100k3C_leiden_eigen'][:npc], axis=1)


In [71]:
coord1 = mcad.obsm['X_leiden_pca'][mcad.obs['Donor']==sample_list[0]][anchor['x1']]
coord2 = mcad.obsm['X_leiden_pca'][mcad.obs['Donor']==sample_list[1]][anchor['x2']]
anchor['dist_leiden_u'] = np.linalg.norm((coord1 - coord2), ord=2, axis=1)


## Integrate with leiden distance on U space

In [72]:
thres1 = np.percentile(anchor['dist_leiden_u'], 80)
thres2 = np.percentile(anchor['dist_leiden_u'], 60)


In [73]:
integrator.anchor[(0,1)] = anchor.loc[anchor['dist_leiden_u']<thres1, ['x1','x2','score']].copy()
corrected = integrator.integrate(key_correct='X_leiden_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [74]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}ldnfilter80'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}ldnfilter80'] = normalize(mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}ldnfilter80'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_u{npc}_seuratcc{npc}ldnfilter80', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}ldnfilter80_tsne'] = mcad.obsm['X_tsne'].copy()


In [75]:
integrator.anchor[(0,1)] = anchor.loc[anchor['dist_leiden_u']<thres2, ['x1','x2','score']].copy()
corrected = integrator.integrate(key_correct='X_leiden_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [76]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}ldnfilter60'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}ldnfilter60'] = normalize(mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}ldnfilter60'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_u{npc}_seuratcc{npc}ldnfilter60', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}ldnfilter60_tsne'] = mcad.obsm['X_tsne'].copy()


In [77]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}ldnfilter60_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['CisLongContact']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [78]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}ldnfilter80_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['CisLongContact']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [79]:
anchor.to_hdf(anchor_file, key='data')


In [80]:
fig, axes = plt.subplots(1, 2, figsize=(5,2), dpi=300)
ax = axes[0]
xx, yy = anchor['dist_leiden_u'], anchor['dist_leiden_pc']
ax.scatter(xx, yy, c='k', edgecolor='none', s=0.5, alpha=1, rasterized=True)
ax.plot([np.min(xx), np.max(xx)], [np.min(xx), np.max(xx)])
ax.set_ylabel('Dist Leiden PC')
ax.set_xlabel('Dist Leiden U')

ax = axes[1]
xx, yy = anchor['dist_pc'], anchor['dist_leiden_pc']
ax.scatter(xx, yy, c='k', edgecolor='none', s=0.5, alpha=1, rasterized=True)
ax.plot([np.min(xx), np.max(xx)], [np.min(xx), np.max(xx)])
ax.set_xlabel('Dist PC')
# ax.set_ylabel('Dist Geosk PC')


In [81]:
# mcad = anndata.AnnData(obs=mcad.obs, obsm=mcad.obsm)
mcad.write_h5ad('HiC_100kb_pca.h5ad')


In [82]:
mcad

## 3c · aggregate refined mCG anchors

_Source: `clustering/tissue/L1/TrCo/04a.5kCG_L2anchor.ipynb`_

In [83]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [84]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [85]:
group_name = 'EG'


In [86]:
# Parameters
cpu = 4
group_name = "TrCo"
mem_gb = 16


In [87]:
mcad = anndata.read_h5ad('mCG_5kb_lsi.h5ad')


In [88]:
mcad

In [89]:
npc = significant_pc_test(mcad, p_cutoff=0.05, update=False, obsm='lsi_all')


In [90]:
mcad.obsm['X_lsi'] = normalize(mcad.obsm['lsi_all'][:, :npc], axis=1)


In [91]:
sample_list = np.sort(mcad.obs['Donor'].unique())
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]


In [92]:
anchor_index = {}
for xx,adata in zip(sample_list, adata_list):
    tmp = pd.DataFrame(index=adata.obs.index).reset_index().reset_index()
    anchor_index[xx] = tmp.set_index('cell')['index'].to_dict()


In [93]:
from glob import glob
indir = f'{ENTEX_ROOT}/'
ct_list = np.sort([xx.split('/')[-2] for xx in glob(f'{indir}clustering/tissue/L2/{group_name}-*/mCG_5kb_seurat_anchor_*.hdf')])
print(len(ct_list))


In [94]:
anchor = []
for ct in ct_list:
    anchor_file = glob(f'{indir}clustering/tissue/L2/{ct}/mCG_5kb_seurat_anchor_*.hdf')[0]
    adata_file = glob(f'{indir}clustering/tissue/L2/{ct}/mCG_5kb_lsi.h5ad')[0]
    anchor_tmp = pd.read_hdf(anchor_file)
    adata_tmp = anndata.read_h5ad(adata_file)
    sample_tmp = np.sort(adata_tmp.obs['Donor'].unique())
    cell_list = [adata_tmp.obs.index[adata_tmp.obs['Donor']==xx] for xx in sample_tmp]
    thres = np.percentile(anchor_tmp['dist_pc'], 90)
    anchor_tmp = anchor_tmp.loc[anchor_tmp['dist_pc']<thres, ['x1','x2','score']].copy()
    anchor_tmp['x1'] = cell_list[0][anchor_tmp['x1']].map(anchor_index[sample_tmp[0]])
    anchor_tmp['x2'] = cell_list[1][anchor_tmp['x2']].map(anchor_index[sample_tmp[1]])
    anchor_tmp['x1_donor'] = sample_tmp[0]
    anchor_tmp['x2_donor'] = sample_tmp[1]
    anchor.append(anchor_tmp)
    
anchor = pd.concat(anchor, axis=0)



In [95]:
anchor[['x1_donor','x2_donor']].value_counts().sort_index()


In [96]:
n = len(adata_list)
anchor_df = {}

for i in range(n - 1):
    for j in range(i + 1, n):
        x1, x2 = sample_list[[i,j]]
        selc = (anchor['x1_donor']==x1) & (anchor['x2_donor']==x2)
        anchor_df[(i,j)] = anchor.loc[selc, ['x1', 'x2', 'score']]
        if selc.sum()==0:
            selc = (anchor['x1_donor']==x2) & (anchor['x2_donor']==x1)
            anchor_df[(i,j)] = anchor.rename({'x1':'x2','x2':'x1'}, axis=1).loc[selc, ['x1', 'x2', 'score']]
        

In [97]:
integrator = SeuratIntegration()


In [98]:
integrator.adata_dict = {k: v for k, v in zip(list(range(len(adata_list))), adata_list)}
integrator.n_dataset = len(adata_list)
integrator.n_cells = [adata.shape[0] for adata in adata_list]

# intra-dataset KNN for scoring the anchors
integrator.k_local = None
integrator.key_local = 'X_lsi'
integrator._calculate_local_knn()

# alignments and all_pairs
integrator.alignments = None
integrator._get_all_pairs()
integrator.anchor = anchor_df


In [99]:
# npc = 50
corrected = integrator.integrate(key_correct='lsi_all',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [100]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'5kCG_u{npc}_seuratL2'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'5kCG_u{npc}_seuratL2'] = normalize(mcad.obsm[f'5kCG_u{npc}_seuratL2'][:, :npc], axis=1)

tsne(mcad, obsm=f'5kCG_u{npc}_seuratL2', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'5kCG_u{npc}_seuratL2_tsne'] = mcad.obsm['X_tsne'].copy()


In [101]:
tmp = anndata.read_h5ad(f'{ENTEX_ROOT}/clustering/merged/cell_86689_16tissue_100k3C_autosomal.h5ad')
mcad.obs['ClusterTissue'] = tmp.obs.loc[mcad.obs.index, 'ClusterTissue'].copy()


In [102]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_u{npc}_seuratL2_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)


ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [103]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6), dpi=300, constrained_layout=True)

for i,method in enumerate(['', f'_seuratL2', '_hm', '_mnn']):

    mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_u{npc}{method}_tsne'].copy()
    dump_embedding(mcad, coord_base)
    tmp = mcad.obs.copy()
    ax = axes[0,i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='Donor',
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab10',
                            scatter_kws={'rasterized':True},
                            show_legend=(i==3))
    ax.set_title(['Raw', 'Seurat', 'Harmony', 'MNN'][i])

    ax = axes[1,i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='ClusterTissue',
                            text_anno='ClusterTissue', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            legend_kws={'ncol':2}, 
                            show_legend=(i==3))


In [104]:
# mcad.obs.index.name = 'cell'
# mcad = anndata.AnnData(obs=mcad.obs, obsm=mcad.obsm)
mcad.write_h5ad('mCG_5kb_lsi.h5ad')


In [105]:
mcad

## 3d · aggregate refined 3C anchors

_Source: `clustering/tissue/L1/TrCo/04b.100k3C_L2anchor.ipynb`_

In [106]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [107]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [108]:
group_name = 'EG'


In [109]:
# Parameters
cpu = 4
group_name = "TrCo"
mem_gb = 16


In [110]:
mcad = anndata.read_h5ad('HiC_100kb_pca.h5ad')


In [111]:
npc = significant_pc_test(mcad, p_cutoff=0.05, update=False, obsm='100k3C_pca')


In [112]:
mcad.obsm['X_pca'] = normalize(mcad.obsm['100k3C_pca'][:, :npc], axis=1)


In [113]:
sample_list = np.sort(mcad.obs['Donor'].unique())
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]


In [114]:
anchor_index = {}
for xx,adata in zip(sample_list, adata_list):
    tmp = pd.DataFrame(index=adata.obs.index).reset_index().reset_index()
    anchor_index[xx] = tmp.set_index('cell')['index'].to_dict()


In [115]:
from glob import glob
indir = f'{ENTEX_ROOT}/'
ct_list = np.sort([xx.split('/')[-2] for xx in glob(f'{indir}clustering/tissue/L2/{group_name}-*/HiC_100kb_seurat_anchor_*.hdf')])
print(len(ct_list))


In [116]:
anchor = []
for ct in ct_list:
    anchor_file = glob(f'{indir}clustering/tissue/L2/{ct}/HiC_100kb_seurat_anchor_*.hdf')[0]
    adata_file = glob(f'{indir}clustering/tissue/L2/{ct}/HiC_100kb_pca.h5ad')[0]
    anchor_tmp = pd.read_hdf(anchor_file)
    adata_tmp = anndata.read_h5ad(adata_file)
    sample_tmp = np.sort(adata_tmp.obs['Donor'].unique())
    cell_list = [adata_tmp.obs.index[adata_tmp.obs['Donor']==xx] for xx in sample_tmp]
    thres = np.percentile(anchor_tmp['dist_pc'], 90)
    anchor_tmp = anchor_tmp.loc[anchor_tmp['dist_pc']<thres, ['x1','x2','score']].copy()
    anchor_tmp['x1'] = cell_list[0][anchor_tmp['x1']].map(anchor_index[sample_tmp[0]])
    anchor_tmp['x2'] = cell_list[1][anchor_tmp['x2']].map(anchor_index[sample_tmp[1]])
    anchor_tmp['x1_donor'] = sample_tmp[0]
    anchor_tmp['x2_donor'] = sample_tmp[1]
    anchor.append(anchor_tmp)
    
anchor = pd.concat(anchor, axis=0)



In [117]:
anchor[['x1_donor','x2_donor']].value_counts().sort_index()


In [118]:
n = len(adata_list)
anchor_df = {}

for i in range(n - 1):
    for j in range(i + 1, n):
        x1, x2 = sample_list[[i,j]]
        selc = (anchor['x1_donor']==x1) & (anchor['x2_donor']==x2)
        anchor_df[(i,j)] = anchor.loc[selc, ['x1', 'x2', 'score']]
        if selc.sum()==0:
            selc = (anchor['x1_donor']==x2) & (anchor['x2_donor']==x1)
            anchor_df[(i,j)] = anchor.rename({'x1':'x2','x2':'x1'}, axis=1).loc[selc, ['x1', 'x2', 'score']]
        

In [119]:
integrator = SeuratIntegration()


In [120]:
integrator.adata_dict = {k: v for k, v in zip(list(range(len(adata_list))), adata_list)}
integrator.n_dataset = len(adata_list)
integrator.n_cells = [adata.shape[0] for adata in adata_list]

# intra-dataset KNN for scoring the anchors
integrator.k_local = None
integrator.key_local = 'X_pca'
integrator._calculate_local_knn()

# alignments and all_pairs
integrator.alignments = None
integrator._get_all_pairs()
integrator.anchor = anchor_df


In [121]:
# npc = 50
corrected = integrator.integrate(key_correct='100k3C_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [122]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_pc{npc}_seuratL2'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_pc{npc}_seuratL2'] = normalize(mcad.obsm[f'100k3C_pc{npc}_seuratL2'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_pc{npc}_seuratL2', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_pc{npc}_seuratL2_tsne'] = mcad.obsm['X_tsne'].copy()


In [123]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_seuratL2_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)


ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['CisLongContact']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [124]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6), dpi=300, constrained_layout=True)

for i,method in enumerate(['', f'_seuratL2', '_hm', '_mnn']):

    mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}{method}_tsne'].copy()
    dump_embedding(mcad, coord_base)
    tmp = mcad.obs.copy()
    ax = axes[0,i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='Donor',
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab10',
                            scatter_kws={'rasterized':True},
                            show_legend=(i==3))
    ax.set_title(['Raw', 'Seurat', 'Harmony', 'MNN'][i])

    ax = axes[1,i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='ClusterTissue',
                            text_anno='ClusterTissue', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            legend_kws={'ncol':2}, 
                            show_legend=(i==3))


In [125]:
# mcad.obs.index.name = 'cell'
# mcad = anndata.AnnData(obs=mcad.obs, obsm=mcad.obsm)
mcad.write_h5ad('HiC_100kb_pca.h5ad')


In [126]:
mcad

## 3e · refined per-tissue joint embedding

_Source: `clustering/tissue/L1/TrCo/05.joint_embed.ipynb`_

In [127]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [128]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [129]:
group_name = 'TrCo'

In [130]:
# Parameters
cpu = 5
group_name = "TrCo"
mem_gb = 20


In [131]:
adata_mc = anndata.read_h5ad('mCG_5kb_lsi.h5ad')
adata_3c = anndata.read_h5ad('HiC_100kb_pca.h5ad')
adata_3c = adata_3c[adata_mc.obs.index].copy()


In [132]:
npc_cg = [int(xx.split('_')[1][1:]) for xx in adata_mc.obsm.keys() if '_seuratL2_tsne' in xx][0]
# npc_3c = [int(xx.split('_')[1][1:]) for xx in adata_3c.obsm.keys() if '100k3C_u' in xx][0]
npc_3c = [int(xx.split('_')[1][2:]) for xx in adata_3c.obsm.keys() if '_seuratL2_tsne' in xx][0]
print(npc_cg, npc_3c)


In [133]:
coord_base = 'tsne'
adata_mc.obsm['X_tsne'] = adata_mc.obsm[f'5kCG_u{npc_cg}_seuratL2_tsne'].copy()
dump_embedding(adata_mc, coord_base)
adata_3c.obsm['X_tsne'] = adata_3c.obsm[f'100k3C_pc{npc_3c}_seuratL2_tsne'].copy()
dump_embedding(adata_3c, coord_base)


In [134]:
adata = anndata.AnnData(obs=adata_mc.obs)
# adata.obsm[f'5kCG100k3C_u{npc_cg}u{npc_3c}'] = np.hstack([normalize(adata_mc.obsm[f'5kCG_u{npc_cg}hm'], axis=1),
#                                                           normalize(adata_3c.obsm['X_pca'], axis=1)])
adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}'] = np.hstack([normalize(adata_mc.obsm[f'5kCG_u{npc_cg}_seuratL2'], axis=1),
                                                           normalize(adata_3c.obsm[f'100k3C_pc{npc_3c}_seuratL2'], axis=1)])
tsne(adata, obsm=f'5kCG100k3C_u{npc_cg}pc{npc_3c}', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}_tsne'] = adata.obsm['X_tsne'].copy()


In [135]:
meta = anndata.read_h5ad(f'{ENTEX_ROOT}/clustering/merged/5kCG100k3C_summary.h5ad').obs
adata.obs['L1_new'] = meta.loc[adata.obs.index, 'L1_annot'].values


In [136]:
ds = 2
coord_base = 'tsne'
adata.obsm[f'X_{coord_base}'] = adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}_{coord_base}'].copy()
dump_embedding(adata, coord_base)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

tmp = adata.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        # palette='tab20',
                        scatter_kws={'rasterized':True},
                        show_legend=True
                       )



ax = axes[0, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[0, 2]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )

ax = axes[1, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='L1_new',
                        text_anno='L1_new', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )

ax = axes[1, 2]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='L1_new',
                        text_anno='L1_new', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )



In [137]:
# clustering name
clustering_name = 'L1'

# ConsensusClustering
# Important factores
n_neighbors = 25
leiden_resolution = 1.0
# this parameter is the final target that limit the total number of clusters
# Higher accuracy means more conservative clustering results and less number of clusters
target_accuracy = 0
min_cluster_size = 30

# Other ConsensusClustering parameters
metric = 'euclidean'
consensus_rate = 0.8
leiden_repeats = 500
random_state = 0
train_frac = 0.5
train_max_n = 500
max_iter = 50
n_jobs = 4

# Dendrogram via Multiscale Bootstrap Resampling
nboot = 10000
method_dist = 'correlation'
method_hclust = 'average'

plot_type = 'static'

In [138]:
cc = ConsensusClustering(model=None,
                         n_neighbors=n_neighbors,
                         metric=metric,
                         min_cluster_size=min_cluster_size,
                         leiden_repeats=leiden_repeats,
                         leiden_resolution=leiden_resolution,
                         consensus_rate=consensus_rate,
                         random_state=random_state,
                         train_frac=train_frac,
                         train_max_n=train_max_n,
                         max_iter=max_iter,
                         n_jobs=n_jobs,
                         target_accuracy=target_accuracy)


In [139]:
cc.fit_predict(adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}'])

In [140]:
adata.obs['leiden_cons'] = cc.label.copy()
adata.obs['leiden_cv'] = cc.cv_predicted_label.copy()


In [141]:
cc.save('ConcensusClustering.model.lib')
adata.write_h5ad('5kCG100k3C_embed.h5ad')


In [142]:
from sklearn.metrics import adjusted_rand_score as ARI
ARI(adata.obs['leiden_cons'], adata.obs['leiden_cv'])


In [143]:
ds = 2
coord_base = 'tsne'
adata.obsm[f'X_{coord_base}'] = adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}_{coord_base}'].copy()
dump_embedding(adata, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = adata.obs.copy()
ax = axes[0,0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='leiden_cons',
                        text_anno='leiden_cons',
                        s=ds,
                        labelsize=12,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0,1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='leiden_cv',
                        text_anno='leiden_cv',
                        s=ds,
                        labelsize=12,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[1,0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='L1_new',
                        text_anno='L1_new', 
                        s=ds,
                        labelsize=12,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )

ax = axes[1,1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=12,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )

# plt.savefig(f'{group_name}_5kCG100k3C_cluster.pdf', transparent=True)


In [144]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12), dpi=300, constrained_layout=True)

for i,xx in enumerate([adata, adata_mc, adata_3c]):
    dump_embedding(xx, coord_base)
    tmp = xx.obs.copy()
    tmp['leiden_cons'] = adata.obs['leiden_cons'].copy()
    tmp['ClusterTissue'] = adata.obs['ClusterTissue'].copy()
    ax = axes[0, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='Donor',
                            s=ds,
                            labelsize=12,
                            max_points=None,
                            # palette='tab20',
                            scatter_kws={'rasterized':True},
                            show_legend=True)

    ax = axes[1, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='leiden_cons',
                            text_anno='leiden_cons', 
                            s=ds,
                            labelsize=12,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            # show_legend=True
                           )

    ax = axes[2, i]
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='ClusterTissue',
                            text_anno='ClusterTissue', 
                            s=ds,
                            labelsize=12,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            # show_legend=False
                           )
